# Simulazione di Gestione Crisi Idrica
## Integrazione Dominio Idrico e Sensoristico

Questo progetto unisce:
- **Dyn-WNTR**: per la simulazione della rete idrica
- **LoRaSim**: per la simulazione della rete di sensori LoRaWAN

**Obiettivo**: Gestire una crisi idrica attraverso serbatoi intelligenti controllati da un agente centrale.

## 1. Import delle Librerie e Configurazione Ambiente

In [ ]:
import sys
import os
import random
import math
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass, field

# Configurazione dei percorsi per le repository
def setup_environment():
    """Configura l'ambiente importando correttamente le due repository."""
    
    # Percorsi delle repository (adattare in base all'ambiente di esecuzione)
    dyn_wntr_path = os.path.join(os.getcwd(), 'Dyn-WNTR')
    lorasim_path = os.path.join(os.getcwd(), 'lorasim')
    
    # Aggiungi i percorsi al sys.path se esistono
    if os.path.exists(dyn_wntr_path) and dyn_wntr_path not in sys.path:
        sys.path.insert(0, dyn_wntr_path)
        print(f"✅ Aggiunto al path: {dyn_wntr_path}")
    
    if os.path.exists(lorasim_path) and lorasim_path not in sys.path:
        sys.path.append(lorasim_path)
        print(f"✅ Aggiunto al path: {lorasim_path}")
    
    # Importa mwntr (Dyn-WNTR)
    try:
        import mwntr
        print(f"✅ Dyn-WNTR importato con successo")
    except ImportError as e:
        print(f"❌ Errore nell'import di Dyn-WNTR: {e}")
        print("Assicurati di aver clonato la repository Dyn-WNTR nella cartella del progetto.")
        raise
    
    # Importa simpy (necessario per LoRaSim)
    try:
        import simpy
        print(f"✅ SimPy importato con successo")
    except ImportError:
        print("⚠️ SimPy non disponibile, installazione in corso...")
        !pip install simpy
        import simpy
    
    return mwntr, simpy

# Esegui la configurazione
mwntr, simpy = setup_environment()

## 2. Classi per la Simulazione LoRaWAN

In [ ]:
@dataclass
class Packet:
    """Rappresenta un pacchetto LoRaWAN."""
    node_id: str
    data: float
    timestamp: float
    sf: int = 7


class Gateway:
    """Gateway LoRaWAN che riceve i pacchetti dai sensori."""
    
    def __init__(self, env: simpy.Environment):
        self.env = env
        self.received_packets: List[Packet] = []
        self.active_transmissions: List[Packet] = []
        self.stat_totale_inviati: int = 0
        self.stat_totale_persi: int = 0
        self.stat_totale_ricevuti: int = 0
    
    def receive_uplink(self, packet: Packet, time_on_air: float):
        """Simula la ricezione di un uplink con possibile collisione."""
        self.stat_totale_inviati += 1
        self.active_transmissions.append(packet)
        
        yield self.env.timeout(time_on_air)
        
        # Rileva collisioni: se più trasmissioni simultanee, tutti i pacchetti vanno persi
        collision = len(self.active_transmissions) > 1
        if not collision:
            self.received_packets.append(packet)
            self.stat_totale_ricevuti += 1
        else:
            self.stat_totale_persi += len(self.active_transmissions)
        
        self.active_transmissions.remove(packet)
    
    def get_packet_loss_rate(self) -> float:
        """Calcola la percentuale di pacchetti persi."""
        if self.stat_totale_inviati == 0:
            return 0.0
        return (self.stat_totale_persi / self.stat_totale_inviati) * 100.0
    
    def get_and_clear_buffer(self) -> List[Packet]:
        """Restituisce e pulisce il buffer dei pacchetti ricevuti."""
        data = self.received_packets.copy()
        self.received_packets.clear()
        return data

In [ ]:
class SensorNode:
    """Nodo sensore che misura pressione e invia dati via LoRaWAN."""
    
    def __init__(self, env: simpy.Environment, node_id: str, gateway: Gateway, 
                 sf: int = 7, tx_interval: int = 3600):
        self.env = env
        self.node_id = node_id
        self.gateway = gateway
        self.sf = sf
        self.tx_interval = tx_interval
        self.current_pressure: float = 0.0
        self.current_tank_level: float = 0.0
        # Time on air dipende dallo Spreading Factor
        self.time_on_air = (2 ** self.sf) / 125000.0 * 20 * 1000  # in ms
        
        # Avvia il processo di trasmissione
        self.env.process(self.transmit_loop())
    
    def update_sensor_data(self, pressure: float, tank_level: float = 0.0):
        """Aggiorna i dati del sensore."""
        self.current_pressure = pressure
        self.current_tank_level = tank_level
    
    def update_tx_interval(self, new_interval: int):
        """Aggiorna l'intervallo di trasmissione."""
        self.tx_interval = new_interval
    
    def transmit_loop(self):
        """Ciclo di trasmissione periodica dei dati."""
        # Ritardo iniziale casuale per evitare sincronizzazione
        yield self.env.timeout(random.uniform(0, min(self.tx_interval, 100)))
        
        while True:
            packet = Packet(
                node_id=self.node_id,
                data=self.current_pressure,
                timestamp=self.env.now,
                sf=self.sf
            )
            
            # Invia il pacchetto attraverso il gateway
            self.env.process(self.gateway.receive_uplink(packet, self.time_on_air))
            
            # Attendi il prossimo intervallo di trasmissione
            yield self.env.timeout(self.tx_interval)

In [ ]:
class LoRaWAN_Network:
    """Rete LoRaWAN che gestisce tutti i nodi sensore."""
    
    def __init__(self, node_ids: List[str], tx_interval: int = 3600):
        self.env = simpy.Environment()
        self.gateway = Gateway(self.env)
        self.nodes: Dict[str, SensorNode] = {}
        
        print("\n📡 INIZIALIZZAZIONE RETE LoRaWAN...")
        for node_id in node_ids:
            # Assegna uno Spreading Factor casuale (7-12)
            sf = random.choice([7, 8, 9, 10, 11, 12])
            node = SensorNode(self.env, node_id, self.gateway, sf, tx_interval)
            self.nodes[node_id] = node
            print(f"   [+] Sensore registrato: {node_id} (SF: {sf})")
    
    def run_communication_step(self, time_step: float) -> List[Packet]:
        """Esegue un passo di simulazione della comunicazione."""
        target_time = self.env.now + time_step
        self.env.run(until=target_time)
        return self.gateway.get_and_clear_buffer()
    
    def update_node_tx_interval(self, node_id: str, new_interval: int):
        """Aggiorna l'intervallo di trasmissione di un nodo specifico."""
        if node_id in self.nodes:
            self.nodes[node_id].update_tx_interval(new_interval)
    
    def get_packet_loss_rate(self) -> float:
        """Restituisce la percentuale attuale di pacchetti persi."""
        return self.gateway.get_packet_loss_rate()

## 3. Classi per la Gestione della Rete Idrica

In [ ]:
@dataclass
class TankConfig:
    """Configurazione per un serbatoio."""
    size_type: str
    tank_diameter: float
    pipe_diameter: float
    min_level: float = 0.0
    max_level: float = 12.0
    init_level: float = 10.0  # Parte quasi pieno


# Configurazioni predefinite per i tre tipi di serbatoi
TANK_CONFIGS = {
    'Small':  TankConfig('Small',  5.0,  0.15, 0.0, 8.0, 6.0),
    'Medium': TankConfig('Medium', 15.0, 0.30, 0.0, 12.0, 10.0),
    'Large':  TankConfig('Large',  40.0, 0.60, 0.0, 15.0, 12.0)
}

In [ ]:
class WaterNetworkManager:
    """Gestisce la rete idrica, inclusi serbatoi, sensori e attuatori."""
    
    def __init__(self, network_file: str):
        """Inizializza il gestore della rete idrica."""
        # Carica il modello della rete
        self.wn = mwntr.network.WaterNetworkModel(network_file)
        print(f"✅ Rete caricata: {network_file}")
        print(f"   Nodi: {self.wn.num_nodes}, Archi: {self.wn.num_links}")
        
        # Traccia i serbatoi IoT aggiunti
        self.iot_tanks: Dict[str, dict] = {}
        self.iot_valves: List[str] = []
    
    def remove_existing_tanks(self):
        """Rimuove tutti i serbatoi esistenti nella rete."""
        original_tanks = list(self.wn.tank_name_list)
        print(f"\n🗑️  Rimozione di {len(original_tanks)} serbatoi esistenti...")
        
        for tank_name in original_tanks:
            # Rimuovi prima i collegamenti al serbatoio
            links_to_remove = list(self.wn.get_links_for_node(tank_name))
            for link_name in links_to_remove:
                self.wn.remove_link(link_name)
            
            # Rimuovi il serbatoio
            self.wn.remove_node(tank_name)
        
        print(f"   ✅ {len(original_tanks)} serbatoi rimossi")
    
    def add_iot_tanks(self, n_tanks: int = 8) -> List[str]:
        """
        Aggiunge serbatoi IoT in posizioni casuali nella rete.
        
        Args:
            n_tanks: Numero di serbatoi da aggiungere
        
        Returns:
            Lista dei nomi delle valvole IoT create
        """
        junctions = self.wn.junction_name_list
        
        if n_tanks > len(junctions):
            print(f"⚠️  Richiesti {n_tanks} serbatoi ma solo {len(junctions)} junction disponibili")
            n_tanks = len(junctions)
        
        # Seleziona nodi casuali per posizionare i serbatoi
        target_nodes = random.sample(junctions, n_tanks)
        tank_types = list(TANK_CONFIGS.keys())
        
        print(f"\n💧 Aggiunta di {n_tanks} serbatoi IoT...")
        
        for i, junc_name in enumerate(target_nodes):
            junc_node = self.wn.get_node(junc_name)
            
            # Scegli un tipo di serbatoio casualmente
            tank_type = random.choice(tank_types)
            config = TANK_CONFIGS[tank_type]
            
            # Calcola posizione e altezza
            offset_height = random.uniform(25, 50)  # Altezza rispetto al nodo
            tank_name = f"IoT_Tank_{tank_type}_{i+1}"
            valve_name = f"IoT_Valve_{i+1}"
            
            # Aggiungi il serbatoio
            self.wn.add_tank(
                name=tank_name,
                elevation=junc_node.elevation + offset_height,
                init_level=config.init_level,
                min_level=config.min_level,
                max_level=config.max_level,
                diameter=config.tank_diameter,
                coordinates=(junc_node.coordinates[0] + 200, 
                           junc_node.coordinates[1] + 200)
            )
            
            # Aggiungi la valvola/pipa di collegamento
            self.wn.add_pipe(
                name=valve_name,
                start_node_name=junc_name,
                end_node_name=tank_name,
                length=50.0,
                diameter=config.pipe_diameter,
                roughness=120,
                initial_status=mwntr.network.LinkStatus.Closed  # Inizialmente chiusa
            )
            
            # Registra il serbatoio IoT
            self.iot_tanks[tank_name] = {
                'type': tank_type,
                'junction': junc_name,
                'valve': valve_name,
                'config': config
            }
            self.iot_valves.append(valve_name)
            
            print(f"   ✅ {tank_name} ({tank_type}) collegato a {junc_name} tramite {valve_name}")
        
        return self.iot_valves

In [ ]:
    def configure_source_pattern(self, pattern_name: str = 'Fonte_Pattern', 
                                  pattern_values: List[float] = None):
        """Configura un pattern per la fonte primaria."""
        if pattern_values is None:
            # Pattern di default: calo progressivo della portata
            pattern_values = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 
                            0.8, 0.6, 0.4, 0.2, 0.1, 0.1, 0.1, 0.1]
        
        self.wn.add_pattern(pattern_name, pattern_values)
        
        # Applica il pattern alla fonte primaria (reservoir)
        reservoirs = list(self.wn.reservoir_name_list)
        if reservoirs:
            primary_source = reservoirs[0]
            source_node = self.wn.get_node(primary_source)
            source_node.head_pattern_name = pattern_name
            print(f"✅ Pattern '{pattern_name}' applicato alla fonte: {primary_source}")
        else:
            print("⚠️  Nessun reservoir trovato nella rete")
    
    def set_simulation_options(self, duration_hours: int = 24, 
                               hydraulic_timestep: int = 300,
                               report_timestep: int = 300):
        """Configura le opzioni di simulazione."""
        self.wn.options.time.duration = duration_hours * 3600
        self.wn.options.time.hydraulic_timestep = hydraulic_timestep
        self.wn.options.time.report_timestep = report_timestep
        
        # Configura il modello di domanda (PDA = Pressure Driven Analysis)
        self.wn.options.hydraulic.demand_model = 'PDA'
        self.wn.options.hydraulic.minimum_pressure = 0.0
        self.wn.options.hydraulic.required_pressure = 20.0
        
        print(f"✅ Simulazione configurata: {duration_hours}h, timestep={hydraulic_timestep}s")
    
    def get_junction_pressure(self, junction_name: str) -> float:
        """Ottiene la pressione attuale in un nodo."""
        try:
            junc = self.wn.get_node(junction_name)
            if hasattr(junc, 'head') and hasattr(junc, 'elevation'):
                return junc.head - junc.elevation
        except Exception:
            pass
        return 0.0
    
    def get_tank_level(self, tank_name: str) -> float:
        """Ottiene il livello attuale di un serbatoio."""
        try:
            tank = self.wn.get_node(tank_name)
            if hasattr(tank, 'level'):
                return tank.level
        except Exception:
            pass
        return 0.0
    
    def set_valve_status(self, valve_name: str, open: bool):
        """Apre o chiude una valvola."""
        try:
            valve = self.wn.get_link(valve_name)
            if open:
                valve.status = mwntr.network.LinkStatus.Opened
            else:
                valve.status = mwntr.network.LinkStatus.Closed
        except Exception as e:
            print(f"⚠️  Errore nel settaggio della valvola {valve_name}: {e}")
    
    def calculate_demand_satisfaction(self) -> float:
        """
        Calcola il livello di soddisfazione della domanda nella rete.
        
        Returns:
            Percentuale di domanda soddisfatta (0-100)
        """
        total_demand = 0.0
        total_flow = 0.0
        
        for junc_name in self.wn.junction_name_list:
            try:
                junc = self.wn.get_node(junc_name)
                if hasattr(junc, 'demand'):
                    total_demand += abs(junc.demand)
                if hasattr(junc, 'flow'):
                    total_flow += abs(junc.flow)
            except Exception:
                continue
        
        if total_demand == 0:
            return 100.0
        
        satisfaction = (total_flow / total_demand) * 100.0
        return min(100.0, max(0.0, satisfaction))

## 4. Agente di Gestione Crisi Idrica

In [ ]:
@dataclass
class AgentState:
    """Stato interno dell'agente decisionale."""
    last_satisfaction: float = 100.0
    action_timestamp: float = 0.0
    tanks_opened: List[str] = field(default_factory=list)
    total_actions: int = 0
    cumulative_reward: float = 0.0


class CrisisManagementAgent:
    """
    Agente intelligente per la gestione della crisi idrica.
    
    Funzione obiettivo:
    - Massimizzare il guadagno in soddisfazione della domanda
    - Minimizzare il tempo di intervento
    - Considerare la qualità della comunicazione (packet loss)
    """
    
    def __init__(self, water_network: WaterNetworkManager, 
                 lora_network: LoRaWAN_Network,
                 satisfaction_threshold: float = 85.0):
        self.wn_manager = water_network
        self.lora_net = lora_network
        self.state = AgentState()
        self.satisfaction_threshold = satisfaction_threshold
        
        # Pesi per la funzione obiettivo
        self.w_gain = 1.0      # Peso per il guadagno in soddisfazione
        self.w_time = 0.1      # Peso per la penalità temporale
        self.w_packet = 0.3    # Peso per la perdita di pacchetti
    
    def compute_objective_function(self, satisfaction_before: float, 
                                   satisfaction_after: float,
                                   time_elapsed: float,
                                   packet_loss_rate: float) -> float:
        """
        Calcola il valore della funzione obiettivo.
        
        Args:
            satisfaction_before: Soddisfazione prima dell'azione
            satisfaction_after: Soddisfazione dopo l'azione
            time_elapsed: Tempo impiegato per realizzare l'azione (secondi)
            packet_loss_rate: Percentuale di pacchetti persi
        
        Returns:
            Valore della funzione obiettivo (da massimizzare)
        """
        # Guadagno in soddisfazione
        gain = satisfaction_after - satisfaction_before
        
        # Penalità temporale (normalizzata)
        time_penalty = time_elapsed / 3600.0  # Normalizza rispetto a un'ora
        
        # Penalità per packet loss
        packet_penalty = packet_loss_rate / 100.0  # Normalizza a [0, 1]
        
        # Funzione obiettivo combinata
        objective = (self.w_gain * gain) - (self.w_time * time_penalty) - (self.w_packet * packet_penalty)
        
        return objective
    
    def decide_tank_activation(self, current_satisfaction: float, 
                               sensor_readings: Dict[str, float]) -> List[str]:
        """
        Decide quali serbatoi aprire in base allo stato della rete.
        
        Strategia semplice:
        - Se la soddisfazione scende sotto la soglia, apri i serbatoi gradualmente
        - Priorità ai serbatoi con livello più alto
        
        Args:
            current_satisfaction: Livello attuale di soddisfazione della domanda
            sensor_readings: Letture dai sensori (pressione per ogni nodo)
        
        Returns:
            Lista delle valvole da aprire
        """
        valves_to_open = []
        
        if current_satisfaction < self.satisfaction_threshold:
            # Ordina i serbatoi per livello decrescente
            tank_levels = []
            for tank_name, tank_info in self.wn_manager.iot_tanks.items():
                level = self.wn_manager.get_tank_level(tank_name)
                valve = tank_info['valve']
                
                # Apri solo se la valvola è chiusa e il serbatoio ha acqua
                if valve not in self.state.tanks_opened and level > 1.0:
                    tank_levels.append((tank_name, level, valve))
            
            # Ordina per livello decrescente
            tank_levels.sort(key=lambda x: x[1], reverse=True)
            
            # Apri i serbatoi in base alla gravità della crisi
            deficit = self.satisfaction_threshold - current_satisfaction
            n_tanks_to_open = min(len(tank_levels), max(1, int(deficit / 10)))
            
            for _, _, valve in tank_levels[:n_tanks_to_open]:
                valves_to_open.append(valve)
        
        return valves_to_open
    
    def adjust_sensor_frequency(self, packet_loss_rate: float, 
                                satisfaction: float) -> int:
        """
        Regola dinamicamente la frequenza di invio dei sensori.
        
        Strategia:
        - Se il packet loss è alto, riduci la frequenza
        - Se la soddisfazione è bassa (crisi), aumenta la frequenza
        
        Args:
            packet_loss_rate: Percentuale di pacchetti persi
            satisfaction: Livello di soddisfazione della domanda
        
        Returns:
            Nuovo intervallo di trasmissione (secondi)
        """
        base_interval = 3600  # 1 ora di base
        
        # Fattore di riduzione per packet loss alto
        if packet_loss_rate > 20:
            interval = base_interval * 2  # Raddoppia l'intervallo
        elif packet_loss_rate > 10:
            interval = base_interval * 1.5
        else:
            interval = base_interval
        
        # Fattore di aumento per crisi idrica
        if satisfaction < 70:
            interval = interval // 4  # Quadruplica la frequenza
        elif satisfaction < 85:
            interval = interval // 2  # Raddoppia la frequenza
        
        return max(300, min(interval, 7200))  # Limita tra 5 min e 2 ore

In [ ]:
    def execute_action(self, valves_to_open: List[str], current_time: float) -> float:
        """
        Esegue l'azione di apertura delle valvole.
        
        Args:
            valves_to_open: Lista delle valvole da aprire
            current_time: Tempo corrente della simulazione
        
        Returns:
            Valore della funzione obiettivo per questa azione
        """
        if not valves_to_open:
            return 0.0
        
        # Registra la soddisfazione prima dell'azione
        satisfaction_before = self.wn_manager.calculate_demand_satisfaction()
        
        # Apri le valvole
        for valve in valves_to_open:
            self.wn_manager.set_valve_status(valve, open=True)
            if valve not in self.state.tanks_opened:
                self.state.tanks_opened.append(valve)
        
        # Esegui un passo di simulazione idraulica per vedere gli effetti
        self.wn_manager.wn.simulate()
        
        # Calcola la soddisfazione dopo l'azione
        satisfaction_after = self.wn_manager.calculate_demand_satisfaction()
        
        # Calcola il tempo impiegato
        time_elapsed = current_time - self.state.action_timestamp
        
        # Ottieni il packet loss rate dalla rete LoRaWAN
        packet_loss = self.lora_net.get_packet_loss_rate()
        
        # Calcola la funzione obiettivo
        objective_value = self.compute_objective_function(
            satisfaction_before,
            satisfaction_after,
            time_elapsed,
            packet_loss
        )
        
        # Aggiorna lo stato
        self.state.last_satisfaction = satisfaction_after
        self.state.action_timestamp = current_time
        self.state.total_actions += 1
        self.state.cumulative_reward += objective_value
        
        print(f"\n🎯 AZIONE ESEGUITA:")
        print(f"   Valvole aperte: {valves_to_open}")
        print(f"   Soddisfazione: {satisfaction_before:.1f}% → {satisfaction_after:.1f}%")
        print(f"   Tempo trascorso: {time_elapsed:.1f}s")
        print(f"   Packet loss: {packet_loss:.1f}%")
        print(f"   Funzione obiettivo: {objective_value:.3f}")
        
        return objective_value

## 5. Co-Simulazione: Integrazione Domini Idrico e Cyber

In [ ]:
class CoSimulationEngine:
    """
    Motore di co-simulazione che coordina dominio fisico (idrico) e cyber (sensori).
    """
    
    def __init__(self, network_file: str, n_iot_tanks: int = 8):
        # Inizializza la rete idrica
        self.water_net = WaterNetworkManager(network_file)
        
        # Rimuovi serbatoi esistenti e aggiungi quelli IoT
        self.water_net.remove_existing_tanks()
        valve_names = self.water_net.add_iot_tanks(n_iot_tanks)
        
        # Configura la simulazione idrica
        self.water_net.set_simulation_options(
            duration_hours=24,
            hydraulic_timestep=300,
            report_timestep=300
        )
        
        # Configura il pattern della fonte primaria
        self.water_net.configure_source_pattern()
        
        # Inizializza la rete LoRaWAN con i sensori associati alle valvole
        self.lora_net = LoRaWAN_Network(valve_names, tx_interval=3600)
        
        # Crea l'agente di gestione crisi
        self.agent = CrisisManagementAgent(self.water_net, self.lora_net)
        
        # Statistiche della simulazione
        self.statistics = {
            'time': [],
            'demand_satisfaction': [],
            'tanks_opened': [],
            'packet_loss_rate': [],
            'objective_value': [],
            'sensor_readings': []
        }
    
    def sync_sensor_data(self):
        """Sincronizza i dati dei sensori con lo stato della rete idrica."""
        for tank_name, tank_info in self.water_net.iot_tanks.items():
            junction = tank_info['junction']
            valve = tank_info['valve']
            
            # Leggi pressione e livello
            pressure = self.water_net.get_junction_pressure(junction)
            level = self.water_net.get_tank_level(tank_name)
            
            # Aggiorna il sensore corrispondente
            if valve in self.lora_net.nodes:
                self.lora_net.nodes[valve].update_sensor_data(pressure, level)
    
    def process_sensor_data(self, packets: List[Packet]) -> Dict[str, float]:
        """Elabora i pacchetti ricevuti dal gateway LoRaWAN."""
        readings = {}
        for packet in packets:
            readings[packet.node_id] = packet.data
        return readings
    
    def run_simulation(self, verbose: bool = True):
        """
        Esegue la co-simulazione completa.
        
        Args:
            verbose: Se True, stampa informazioni dettagliate
        """
        if verbose:
            print("\n" + "="*70)
            print("🚀 AVVIO CO-SIMULAZIONE")
            print("="*70)
        
        # Parametri di simulazione
        total_duration = self.water_net.wn.options.time.duration  # in secondi
        hydraulic_step = self.water_net.wn.options.time.hydraulic_timestep
        communication_step = 60  # Ogni 60 secondi simulo le comunicazioni
        
        current_time = 0.0
        
        while current_time < total_duration:
            # 1. Sincronizza i sensori con lo stato attuale della rete
            self.sync_sensor_data()
            
            # 2. Esegui un passo di comunicazione LoRaWAN
            packets = self.lora_net.run_communication_step(communication_step)
            sensor_readings = self.process_sensor_data(packets)
            
            # 3. Esegui un passo di simulazione idraulica
            try:
                self.water_net.wn.simulate()
            except Exception as e:
                if verbose:
                    print(f"⚠️  Warning simulazione idraulica: {e}")
            
            # 4. Calcola metriche attuali
            satisfaction = self.water_net.calculate_demand_satisfaction()
            packet_loss = self.lora_net.get_packet_loss_rate()
            n_tanks_open = len(self.agent.state.tanks_opened)
            
            # 5. L'agente prende decisioni
            if current_time > 0 and int(current_time) % 1800 == 0:  # Ogni 30 minuti
                valves_to_open = self.agent.decide_tank_activation(satisfaction, sensor_readings)
                
                if valves_to_open:
                    obj_value = self.agent.execute_action(valves_to_open, current_time)
                    
                    # Regola la frequenza dei sensori
                    new_interval = self.agent.adjust_sensor_frequency(packet_loss, satisfaction)
                    for node_id in self.lora_net.nodes:
                        self.lora_net.update_node_tx_interval(node_id, new_interval)
                else:
                    obj_value = 0.0
            else:
                obj_value = 0.0
            
            # 6. Registra le statistiche
            self.statistics['time'].append(current_time)
            self.statistics['demand_satisfaction'].append(satisfaction)
            self.statistics['tanks_opened'].append(n_tanks_open)
            self.statistics['packet_loss_rate'].append(packet_loss)
            self.statistics['objective_value'].append(obj_value)
            self.statistics['sensor_readings'].append(sensor_readings)
            
            # 7. Stampa progresso
            if verbose and int(current_time) % 3600 == 0:
                hours = current_time / 3600
                print(f"\n⏰ Ora {hours:.1f}h: Soddisfazione={satisfaction:.1f}%, "
                      f"Serbatoi aperti={n_tanks_open}, Packet Loss={packet_loss:.1f}%")
            
            # Avanza il tempo
            current_time += communication_step
        
        if verbose:
            print("\n" + "="*70)
            print("✅ SIMULAZIONE COMPLETATA")
            print("="*70)
            print(f"\n📊 RISULTATI FINALI:")
            print(f"   Azioni totali eseguite: {self.agent.state.total_actions}")
            print(f"   Reward cumulativo: {self.agent.state.cumulative_reward:.3f}")
            print(f"   Packet loss medio: {sum(self.statistics['packet_loss_rate'])/len(self.statistics['packet_loss_rate']):.1f}%")
            print(f"   Soddisfazione media: {sum(self.statistics['demand_satisfaction'])/len(self.statistics['demand_satisfaction']):.1f}%")

## 6. Esecuzione della Simulazione

In [ ]:
# Imposta il seed per riproducibilità
random.seed(42)

# Verifica se il file Net4 esiste
network_file = 'Net4.inp'

if not os.path.exists(network_file):
    print(f"⚠️  File {network_file} non trovato!")
    print("Cerco file .inp nella directory corrente...")
    inp_files = [f for f in os.listdir('.') if f.endswith('.inp')]
    if inp_files:
        network_file = inp_files[0]
        print(f"Usato file alternativo: {network_file}")
    else:
        print("❌ Nessun file .inp trovato. Assicurati di avere un file di rete idrica.")
        network_file = None

# Esegui la co-simulazione se il file esiste
if network_file:
    print(f"\n🔧 Preparazione simulazione con rete: {network_file}")
    
    # Crea e avvia il motore di co-simulazione
    engine = CoSimulationEngine(network_file, n_iot_tanks=8)
    engine.run_simulation(verbose=True)
    
    # Mostra un riepilogo finale
    print("\n" + "="*70)
    print("📈 ANALISI POST-SIMULAZIONE")
    print("="*70)
    
    import numpy as np
    stats = engine.statistics
    
    print(f"\n📊 STATISTICHE CHIAVE:")
    print(f"   • Durata simulazione: {max(stats['time'])/3600:.1f} ore")
    print(f"   • Soddisfazione massima: {max(stats['demand_satisfaction']):.1f}%")
    print(f"   • Soddisfazione minima: {min(stats['demand_satisfaction']):.1f}%")
    print(f"   • Soddisfazione media: {np.mean(stats['demand_satisfaction']):.1f}%")
    print(f"   • Packet loss massimo: {max(stats['packet_loss_rate']):.1f}%")
    print(f"   • Numero massimo serbatoi aperti: {max(stats['tanks_opened'])}")
    print(f"   • Reward medio per azione: {np.mean([v for v in stats['objective_value'] if v != 0]):.3f}" if any(v != 0 for v in stats['objective_value']) else "   • Nessuna azione eseguita")
    
    print(f"\n✅ Simulazione completata con successo!")

## 7. Visualizzazione Risultati (Opzionale)

In [ ]:
# Cell opzionale per visualizzazione grafica (richiede matplotlib)
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10))
    
    # Grafico 1: Soddisfazione della domanda nel tempo
    axes[0].plot([t/3600 for t in engine.statistics['time']], 
                 engine.statistics['demand_satisfaction'], 
                 'b-', linewidth=2, label='Domanda Soddisfatta (%)')
    axes[0].axhline(y=engine.agent.satisfaction_threshold, color='r', linestyle='--', 
                   label=f'Soglia ({engine.agent.satisfaction_threshold}%)')
    axes[0].set_xlabel('Tempo (ore)')
    axes[0].set_ylabel('Soddisfazione (%)')
    axes[0].set_title('Andamento Soddisfazione della Domanda')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Grafico 2: Serbatoi aperti nel tempo
    axes[1].plot([t/3600 for t in engine.statistics['time']], 
                 engine.statistics['tanks_opened'], 
                 'g-', linewidth=2, label='Serbatoi Aperti')
    axes[1].set_xlabel('Tempo (ore)')
    axes[1].set_ylabel('Numero Serbatoi')
    axes[1].set_title('Attivazione Serbatoi IoT')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Grafico 3: Packet Loss Rate
    axes[2].plot([t/3600 for t in engine.statistics['time']], 
                 engine.statistics['packet_loss_rate'], 
                 'orange', linewidth=2, label='Packet Loss (%)')
    axes[2].set_xlabel('Tempo (ore)')
    axes[2].set_ylabel('Packet Loss (%)')
    axes[2].set_title('Qualità Comunicazione LoRaWAN')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('simulation_results.png', dpi=300, bbox_inches='tight')
    print("\n📊 Grafici salvati in 'simulation_results.png'")
    plt.show()
    
except ImportError:
    print("⚠️ Matplotlib non disponibile. Installa con: pip install matplotlib")